In [1]:
from math import exp, sqrt


def arbol_americano(precio_inicial, strike, tasa, volatilidad, vencimiento,
                     pasos, tipo_opcion, subyacente):
    """Precio mediante árbol binomial CRR y ejercicio anticipado."""
    dt = vencimiento / pasos
    u = exp(volatilidad * sqrt(dt))
    d = 1 / u

    # El futuro tiene crecimiento neutral al riesgo igual a 1.
    crecimiento = exp(tasa * dt) if subyacente == "accion" else 1.0
    p = (crecimiento - d) / (u - d)
    descuento = exp(-tasa * dt)

    def ejercicio(precio):
        if tipo_opcion == "put":
            return max(strike - precio, 0)
        return max(precio - strike, 0)

    # Al vencimiento: j indica cuántas subidas ocurrieron.
    valores = [ejercicio(precio_inicial * u**j * d**(pasos - j))
               for j in range(pasos + 1)]
    niveles = [None] * (pasos + 1)
    niveles[pasos] = valores.copy()
    ejercicios_anticipados = []

    # Retroceder hasta el momento inicial.
    for i in range(pasos - 1, -1, -1):
        anteriores = []
        for j in range(i + 1):
            precio = precio_inicial * u**j * d**(i - j)
            continuar = descuento * ((1 - p) * valores[j] + p * valores[j + 1])
            ejercer = ejercicio(precio)
            anteriores.append(max(continuar, ejercer))
            if ejercer > continuar + 1e-10:
                ejercicios_anticipados.append((i, precio, ejercer))
        valores = anteriores
        niveles[i] = valores.copy()

    print(f"u = {u:.6f}, d = {d:.6f}, p = {p:.6f}")
    print("Árbol de valores (de precio bajo a precio alto del subyacente):")
    for i, nivel in enumerate(niveles):
        print(f"Paso {i}: " + " | ".join(f"{v:.4f}" for v in nivel))
    print(f"Precio de la opción americana: {valores[0]:.4f}")
    if ejercicios_anticipados:
        print("Existe ejercicio anticipado óptimo en al menos un nodo.")
    else:
        print("No se identificó ejercicio anticipado óptimo en el árbol.")
    return valores[0]

In [2]:
# %% Ejercicio 21.2: put americana sobre una acción
print("EJERCICIO 21.2 - Resultado en dólares por acción")
precio_put = arbol_americano(
    precio_inicial=60, strike=60, tasa=0.10, volatilidad=0.45,
    vencimiento=3/12, pasos=3, tipo_opcion="put", subyacente="accion"
)
print()

EJERCICIO 21.2 - Resultado en dólares por acción
u = 1.138719, d = 0.878180, p = 0.499688
Árbol de valores (de precio bajo a precio alto del subyacente):
Paso 0: 5.1628
Paso 1: 8.6084 | 1.7993
Paso 2: 13.7280 | 3.6265 | 0.0000
Paso 3: 19.3649 | 7.3092 | 0.0000 | 0.0000
Precio de la opción americana: 5.1628
Existe ejercicio anticipado óptimo en al menos un nodo.



In [3]:
# %% Ejercicio 21.4: call americana sobre futuros de maíz
print("EJERCICIO 21.4 - Resultado en centavos por unidad")
precio_call = arbol_americano(
    precio_inicial=198, strike=200, tasa=0.08, volatilidad=0.30,
    vencimiento=9/12, pasos=3, tipo_opcion="call", subyacente="futuro"
)

EJERCICIO 21.4 - Resultado en centavos por unidad
u = 1.161834, d = 0.860708, p = 0.462570
Árbol de valores (de precio bajo a precio alto del subyacente):
Paso 0: 20.3371
Paso 1: 6.1763 | 37.6777
Paso 2: 0.0000 | 13.6219 | 67.2720
Paso 3: 0.0000 | 0.0000 | 30.0432 | 110.5258
Precio de la opción americana: 20.3371
Existe ejercicio anticipado óptimo en al menos un nodo.
